# 5-OTA Example

## Python Setup

In [1]:
from pygmid import Lookup as lk
import numpy as np
from IPython.display import HTML, display

nmos = lk('../mat/nfet_01v8_lvt.mat')
pmos = lk('../mat/pfet_01v8_lvt.mat')

## Input Parameters

In [2]:
# Circuit Parameters
vdd = 1.8 # V
ugf = 50e6 # Hz
cload = 1e-12 # F

# Device Parameters
## gm/ID
gm_id_M12 = 20
gm_id_M34 = 6
gm_id_M56 = 10

## Lengths
l_M12 = 0.5
l_M34 = 1
l_M56 = 1

## Calculations

In [ ]:
# Overdrive Voltages (Vdsat [V])
vdsat_M12 = 2 / gm_id_M12
vdsat_M34 = 2 / gm_id_M34
vdsat_M56 = 2 / gm_id_M56

# Gate-Source Voltages (Vgs [V])
vgs_M12 = nmos.lookupVGS(GM_ID=gm_id_M12, VDS=vdd/2, VSB=vdsat_M56, L=l_M12)
vgs_M34 = pmos.lookupVGS(GM_ID=gm_id_M34, VSB=0, L=l_M34)
vgs_M34 = pmos.lookupVGS(GM_ID=gm_id_M34, VDS=vgs_M34, VSB=0, L=l_M34)
vgs_M56 = nmos.lookupVGS(GM_ID=gm_id_M12, VDS=vdsat_M56, VSB=0, L=l_M56)

# 'Optimize' Loop
cself = 0
for i in range(20):
    # M1/2 Transconductance (gm [S]) & Current (ID [A])
    gm_M12 = 2 * np.pi * (cload + cself) * ugf
    id_M12 = id_M34 = gm_M12 / gm_id_M12

    # M3/4 Transconductance (gm [S])
    gm_M34 = gm_id_M34 * id_M34

    # M1/2 & M3/4 Current Density (JD [µA/m])
    jd_M12 = nmos.lookup('id_w', GM_ID=gm_id_M12, VDS=vdd/2, VSB=vdsat_M56, L=l_M12)
    jd_M34 = pmos.lookup('id_w', GM_ID=gm_id_M34, VDS=vgs_M34, VSB=0, L=l_M34)

    # M1/2 & M3/4 Width (W [µm])
    w_M12 = id_M12 / jd_M12
    w_M34 = id_M34 / jd_M34

    # M1/2 & M3/4 Capacitance (CDD [F])
    cdd_M12 = nmos.lookup('CDD_w', GM_ID=gm_id_M12, VDS=vdd/2, VSB=vdsat_M56, L=l_M12) * w_M12
    cdd_M34 = pmos.lookup('CDD_w', GM_ID=gm_id_M34, VDS=vgs_M34, VSB=0, L=l_M34) * w_M34
    cself = (cdd_M12 + cdd_M34)

# M5/6 - Tail Current, Transconductance, Current Density & Width
id_M56 = 2 * id_M12
gm_M56 = gm_id_M56 * id_M56
jd_M56 = nmos.lookup('id_w', GM_ID=gm_id_M56, VDS=vdsat_M56, VSB=0, L=l_M56)
w_M56 = id_M56 / jd_M56

# Intrinsic Gains (gm/gds [V/V])
av_M12 = nmos.lookup('GM_GDS', GM_ID=gm_id_M12, VDS=vdd/2, VSB=vdsat_M56, L=l_M12)
av_M34 = pmos.lookup('GM_GDS', GM_ID=gm_id_M34, VDS=vgs_M34, VSB=0, L=l_M34)
av_M56 = nmos.lookup('GM_GDS', GM_ID=gm_id_M56, VDS=vdsat_M56, VSB=0, L=l_M56)

# Output Conductances (gds [S])
gds_M12 = gm_M12 / av_M12
gds_M34 = gm_M34 / av_M34
gds_M56 = gm_M56 / av_M56

# Threshold Voltages (Vth [V])
vth_M12 = nmos.lookup('VT', vgs=vgs_M12, VDS=vdd/2, VSB=vdsat_M56, L=l_M12)
vth_M34 = pmos.lookup('VT', vgs=vgs_M34, VDS=vgs_M34, VSB=0, L=l_M34)
vth_M56 = nmos.lookup('VT', vgs=vgs_M56, VDS=vgs_M56, VSB=0, L=l_M56)

# Circuit Gain (A0 [V/V, dB])
a0_Mag = gm_M12 / (gds_M12 + gds_M34)
a0_dB = 20 * np.log10(a0_Mag)

# Circuit Consumption (IDD [A])
idd = id_M56

# Circuit Unity-Gain Frequency (UGF [Hz])
ugf = gm_M12 / (2 * np.pi * (cload + cself))

## Output

In [ ]:
html_output = f"""
<style>
 .param-title {{ font-size: 18px; font-weight: bold; margin: 20px 0 10px 0; }}
 .param-table {{ border-collapse: collapse; margin: 20px 0; font-size: 14px; }}
 .param-table th, .param-table td {{ border: 1px solid #ddd; padding: 8px 12px; text-align: right; }}
 .param-table th {{ background-color: #000000; color: white; font-weight: bold; }}
 .param-table td:first-child {{ text-align: left; font-weight: 500; }}
 .param-table td:last-child {{ text-align: center; }}
 .circuit-params {{ font-family: monospace; margin: 20px 0; line-height: 1.6; }}
</style>
<div class="param-title">MOSFET PARAMETERS:</div>
<table class="param-table">
 <tr>
 <th>Parameter</th><th>M1/2</th><th>M3/4</th><th>M5/6</th><th>Unit</th>
 </tr>
 <tr><td>W</td><td>{w_M12:.3f}</td><td>{w_M34:.3f}</td><td>{w_M56:.3f}</td><td>µm</td></tr>
 <tr><td>L</td><td>{l_M12:.3f}</td><td>{l_M34:.3f}</td><td>{l_M56:.3f}</td><td>µm</td></tr>
 <tr><td>gm/ID</td><td>{gm_id_M12:.3f}</td><td>{gm_id_M34:.3f}</td><td>{gm_id_M56:.3f}</td><td>S/A</td></tr>
 <tr><td>JD</td><td>{jd_M12/1e-6:.3f}</td><td>{jd_M34/1e-6:.3f}</td><td>{jd_M56/1e-6:.3f}</td><td>µA/µm</td></tr>
 <tr><td>gm/gds</td><td>{av_M12:.3f}</td><td>{av_M34:.3f}</td><td>{av_M56:.3f}</td><td>V/V</td></tr>
 <tr><td>gm</td><td>{gm_M12/1e-6:.3f}</td><td>{gm_M34/1e-6:.3f}</td><td>{gm_M56/1e-6:.3f}</td><td>µS</td></tr>
 <tr><td>ID</td><td>{id_M12/1e-6:.3f}</td><td>{id_M34/1e-6:.3f}</td><td>{id_M56/1e-6:.3f}</td><td>µA</td></tr>
 <tr><td>gds</td><td>{gds_M12/1e-6:.3f}</td><td>{gds_M34/1e-6:.3f}</td><td>{gds_M56/1e-6:.3f}</td><td>µS</td></tr>
 <tr><td>Vdsat</td><td>{vdsat_M12/1e-3:.3f}</td><td>{vdsat_M34/1e-3:.3f}</td><td>{vdsat_M56/1e-3:.3f}</td><td>mV</td></tr>
 <tr><td>Vth</td><td>{vth_M12/1e-3:.3f}</td><td>{vth_M34/1e-3:.3f}</td><td>{vth_M56/1e-3:.3f}</td><td>mV</td></tr>
 <tr><td>Vgs</td><td>-</td><td>{vgs_M34/1e-3:.3f}</td><td>{vgs_M56/1e-3:.3f}</td><td>mV</td></tr>
</table>
<div class="param-title">CIRCUIT PARAMETERS:</div>
<div class="circuit-params">
A0 = {a0_dB:.3f} dB, {a0_Mag:.3f} V/V<br>
UGF ≈ {ugf/1e6:.3f} MHz<br>
IDD = {idd/1e-6:.3f} µA<br>
CL = {cload/1e-12:.3f} pF
</div>
"""
display(HTML(html_output))

Parameter,M1/2,M3/4,M5/6,Unit
w,16.383,5.217,7.991,µm
L,0.500,1.000,1.000,µm
gm/ID,20.000,6.000,10.000,S/A
JD,0.971,3.048,3.980,µA/µm
gm/gds,89.917,55.809,12.860,V/V
gm,318.044,95.413,318.044,µS
ID,15.902,15.902,31.804,µA
gds,3.537,1.710,24.730,µS
Vdsat,100.000,333.333,200.000,mV
Vth,554.200,444.260,483.000,mV
